## Computer Vision - Part 6

### Robot Kinematics - Mapping

In robotics, kinematics is the study of motion without considering the forces that cause it. Mapping in robot kinematics refers to the process of determining the position and orientation of a robot in its environment. This is crucial for tasks such as navigation, manipulation, and interaction with objects.

This lab is structured as five focused modules, each building on the last. By the end, you'll have a complete pipeline from a camera-detected object pose to a robot-ready target transform.

SE(3) is the mathematical group that describes the combined space of rotations and translations in three-dimensional space. It is fundamental in robotics for representing the pose of a robot or an object, which includes both its position (translation) and orientation (rotation).

The SpatialMath library provides tools for working with SE(3) transformations, allowing you to easily manipulate and combine rotations and translations. This is essential for tasks such as mapping, where you need to determine the robot's position and orientation in relation to its environment.

- Rotation: Create rotations around X, Y, Z axes using `SE3.Rx()`, `SE3.Ry()`, `SE3.Rz()` with angles in radians.

- Translation: Build translations using `SE3.Tx()`, `SE3.Ty()`, `SE3.Tz()` and compose them with the * operator.

- Inverse Check: Verify that `T.inv() * T * p ≈ p`. This sanity check confirms your transform is valid and reversible.

### SE(3) in Action: Rotation + Translation + Inverse

Run this starter code, then create at least 3 more transform examples on your own. Try combining Rx, Ry, Tz in different orders — notice how order matters!

In [4]:
!pip install spatialmath-python


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
from spatialmath import SE3
import numpy as np

# Rotation 45° around Z axis
Rz = SE3.Rz(np.deg2rad(45))

# Translation: 0.2m along X, 0.1m along Y
T_trans = SE3.Tx(0.2) * SE3.Ty(0.1)

# Compose: translate THEN rotate
T = T_trans * Rz
p = np.array([0.1, 0.0, 0.0]) # Point 10cm on X-axis
p_new = T * p

print("T =\n", T.A)
print("p_new =", p_new)
# Inverse check: T_inv * (T * p) should equal p
p_recovered = T.inv() * p_new
print("Recovered p =", p_recovered) # Should match original p

T =
 [[ 0.70710678 -0.70710678  0.          0.2       ]
 [ 0.70710678  0.70710678  0.          0.1       ]
 [ 0.          0.          1.          0.        ]
 [ 0.          0.          0.          1.        ]]
p_new = [[0.27071068]
 [0.17071068]
 [0.        ]]
Recovered p = [[ 1.00000000e-01]
 [-2.77555756e-17]
 [ 0.00000000e+00]]


Key insight: Transform composition is NOT commutative. `T_trans * Rz ≠ Rz * T_trans`. Try swapping the order and see what changes!

In [6]:
T2 = Rz * T_trans
p_new = T2 * p
print("T2 =\n", T2.A)
print("p_new =", p_new)
# Inverse check: T_inv * (T * p) should equal p
p_recovered = T.inv() * p_new
print("Recovered p =", p_recovered) # Should match original p

T2 =
 [[ 0.70710678 -0.70710678  0.          0.07071068]
 [ 0.70710678  0.70710678  0.          0.21213203]
 [ 0.          0.          1.          0.        ]
 [ 0.          0.          0.          1.        ]]
p_new = [[0.14142136]
 [0.28284271]
 [0.        ]]
Recovered p = [[0.08786797]
 [0.17071068]
 [0.        ]]


#### Frame Change: Camera → Robot Base

After hand-eye calibration, we know `T_base_cam` — the fixed transform from camera frame to robot base frame.

Given an object's pose in camera frame `T_cam_obj`, the object in base frame is:

`T_base_obj = T_base_cam × T_cam_obj`

This is the chain rule of transforms — the core operation you'll use in every robotics pipeline.

`T_cam_obj` refers to the object's 6D pose relative to the camera. But the robot only understands positions in its own base frame. This conversion is the essential bridge between vision and
manipulation.

- Camera frame: where the object appears
- Base frame: where the robot needs to go
- `T_base_cam`: the calibrated bridge between them

Deliverable: Implement `to_base_frame(T_base_cam, T_cam_obj)` as a reusable function.

In [7]:
# Frame Conversion
from spatialmath import SE3
import numpy as np

# Camera is 0.3m above base, rotated 90° around Y
# (This comes from your hand-eye calibration result)
T_base_cam = SE3.Tz(0.3) * SE3.Ry(np.deg2rad(90))
# Object detected in camera frame: 15cm forward, 5cm up, rotated 30° around Z
T_cam_obj = SE3.Tx(0.15) * SE3.Tz(0.05) * SE3.Rz(np.deg2rad(30))
# Transform to base frame — the key operation
T_base_obj = T_base_cam * T_cam_obj
print("Object in base frame:\n", T_base_obj.A)

# This is the pose you would use for motion planning to grasp the object.
def to_base_frame(T_base_cam, T_cam_obj):
    # Convert object pose from camera frame to robot base frame.
    return T_base_cam * T_cam_obj

# Test it
result = to_base_frame(T_base_cam, T_cam_obj)
print("Via function:\n", result.A)

Object in base frame:
 [[ 5.30287619e-17 -3.06161700e-17  1.00000000e+00  5.00000000e-02]
 [ 5.00000000e-01  8.66025404e-01  0.00000000e+00  0.00000000e+00]
 [-8.66025404e-01  5.00000000e-01  6.12323400e-17  1.50000000e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
Via function:
 [[ 5.30287619e-17 -3.06161700e-17  1.00000000e+00  5.00000000e-02]
 [ 5.00000000e-01  8.66025404e-01  0.00000000e+00  0.00000000e+00]
 [-8.66025404e-01  5.00000000e-01  6.12323400e-17  1.50000000e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


#### PoE & Forward Kinematics via Toolbox

Instead of implementing Product of Exponentials from scratch, we use roboticstoolbox as a practical demonstration. This library provides built-in functions for forward kinematics, allowing you to compute the end-effector pose given joint angles and robot parameters.

- Choose a Robot Model: Use `rtb.models.ETS2.planar2()` — a simple 2-DOF planar robot. If your lab has a Yahboom model, try that instead for real-world relevance.

- Vary Joint Configs: Try `q = [0, 0], [π/2, 0], [0, π/2]`, and combinations. Watch how `fkine(q)` changes the TCP transform.

- Read the Output: The 4×4 matrix from `T_fk.` is an SE(3) transform. The top-left 3×3 is rotation; the right column (xyz) is position 

To avoid version conflicts, ensure you have the latest version of roboticstoolbox installed. You can install it with powershell

- Create a temporary virtual environment to isolate dependencies. Then install roboticstoolbox using pip and numpy< 2.0 to avoid compatibility issues.

- Remember to activate the virtual environment before running your code.

- After running all the codes, you can deactivate the virtual environment and remove it if you no longer need it.

In [8]:
!python -m venv temp_venv
!temp_venv/bin/pip install "numpy<2" roboticstoolbox-python ipykernel
!temp_venv/bin/python -m ipykernel install --user --name=temp_venv --display-name "Python (temp_venv)"


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Installed kernelspec temp_venv in /Users/macbook/Library/Jupyter/kernels/temp_venv


In [9]:
import sys
import numpy as np

print(sys.executable)
print(np.__version__)

/Users/macbook/Documents/Jupyter Notebook/temp_venv/bin/python
1.26.4


In [11]:
# FK with roboticstoolbox
import roboticstoolbox as rtb
import numpy as np

# Load a simple 2-DOF planar robot
robot = rtb.models.DH.Planar2()
# Try different joint configurations
configs = {"home": np.array([0.0, 0.0]), 
           "elbow": np.array([0.5, 0.5]), 
           "reach": np.array([0.2, -0.3]), }

for name, q in configs.items():
    T_fk = robot.fkine(q)
    print(f"\nConfig '{name}': q = {q}")
    print("TCP pose T_fk =\n", T_fk.A)


Config 'home': q = [0. 0.]
TCP pose T_fk =
 [[1. 0. 0. 2.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]

Config 'elbow': q = [0.5 0.5]
TCP pose T_fk =
 [[ 0.54030231 -0.84147098  0.          1.41788487]
 [ 0.84147098  0.54030231  0.          1.32089652]
 [ 0.          0.          1.          0.        ]
 [ 0.          0.          0.          1.        ]]

Config 'reach': q = [ 0.2 -0.3]
TCP pose T_fk =
 [[ 0.99500417  0.09983342  0.          1.97507074]
 [-0.09983342  0.99500417  0.          0.09883591]
 [ 0.          0.          1.          0.        ]
 [ 0.          0.          0.          1.        ]]


- How does changing `q[0]` alone affect TCP position? The rotation of the first joint will cause the entire arm to swing, changing the TCP's position in a circular arc around the base.

- When is the arm fully extended? The arm is fully extended when both joints are at 0 degrees (`q = [0, 0]`), resulting in the TCP being at its maximum reach from the base.

- Can you find a config where the TCP is back near the base? Try `q = [π/2, π/2]` — the arm folds back on itself, bringing the TCP close to the base.

#### Jacobian & Resolved-Rate Control (Preview)

The Jacobian `J(q)` maps joint velocities to TCP velocities: `ẋ = J(q) · q̇` . Inverting this lets us ask: "what joint speeds produce the TCP motion I want?" This is the foundation of velocity-level robot control.

- Compute J(q): Use `robot.jacob0(q)` to get the Jacobian at your current joint config. For `planar2`, this will be a 2x2 matrix, each column representing how one joint's velocity affects the TCP's x and y velocities.

- Use `np.linalg.pinv(J)` to compute J⁺ (the pseudoinverse) of the Jacobian. This allows you to solve for the desired TCP velocity to get joint velocities: `dq = J⁺ · ẋ`. This approach works even when J is not square.

- You don't need to prove where J comes from today. Focus on the result: inspect dq and ask yourself which joint moves more when TCP moves along X vs. Y.

In [16]:
import roboticstoolbox as rtb
import numpy as np
robot = rtb.models.DH.Planar2()

q = np.array([0.5, 0.3])
J = robot.jacob0(q)

J_xy = J[0:2, :]  # extract planar Jacobian
v = np.array([0.01, 0.0])  # desired x velocity

dq = np.linalg.solve(J_xy, v)

print("J_xy =\n", J_xy)
print("dq =", dq)

J_xy =
 [[-1.19678163 -0.71735609]
 [ 1.57428927  0.69670671]]
dq = [ 0.0235756 -0.0532718]


In [17]:
!jupyter kernelspec uninstall -f temp_venv
!rm -rf temp_venv

Removed /Users/macbook/Library/Jupyter/kernels/temp_venv


### Recap

Full Pipeline: Vision → Pose → Robot Target

- Vision gives us `T_cam_obj`
- Hand-eye calibration gives us `T_base_cam`
- We compute `T_base_obj = T_base_cam × T_cam_obj`
- Forward kinematics maps joint angles to TCP pose
- Jacobian allows us to control TCP velocity via joint velocities

In [ ]:
''' from spatialmath import SE3
import numpy as np
# Step 1: Known calibration (hand-eye result)
T_base_cam = SE3.Tz(0.3) * SE3.Ry(np.deg2rad(90))
# Step 2: Object pose from Week 5 vision (placeholder)
T_cam_obj = SE3.Tx(0.15) * SE3.Tz(0.05) * SE3.Rz(np.deg2rad(30))
# Step 3: Convert to base frame
T_base_obj = T_base_cam * T_cam_obj
# Step 4: Define grasp target (Week 8 will add tool offset)
T_target = T_base_obj
print("T_target (input for IK in Week 8):\n", T_target.A)
'''